# Top-P 采样（Nucleus Sampling，Holtzman et al. 2019）

> 本文件原为空，按"详细解析 + 骨架"补全。

## 1. 原理
1. 按概率降序排序；
2. 累积概率，保留**累积刚超过 p** 的最小前缀集合（nucleus）；
3. 在该集合上归一化采样。

## 2. 与 TopK 对比
- TopK：集合大小固定 K；TopP：集合大小**动态**——分布尖锐时集合小（更确定），分布平坦时集合大（更多样）。
- p 常取 0.9~0.95；p→1 退化为纯采样，p→0 退化为贪心。
- 实践常 TopP + TopK 同时用（先 TopK 再 TopP）。

## 3. 考察点
- `sort` 降序、`cumsum`、`searchsorted` 找截断点
- 把 nucleus 外概率置 0 再归一化
- 与温度组合顺序

In [ ]:
import torch
import torch.nn.functional as F

def topp_sampling(logits, p=0.9, temperature=1.0):
    """
    logits: [batch, vocab] 或 [vocab]
    返回采样到的 token id。
    """
    # TODO:
    # probs = F.softmax(logits / temperature, dim=-1)
    # sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
    # cum = torch.cumsum(sorted_probs, dim=-1)
    # 保留累积到刚超过 p 的那一个（即 cum <= p 的全部 + 第一个超过 p 的）
    # mask = cum - sorted_probs > p        # 这些是要置零的（在 sorted 顺序里）
    # sorted_probs = sorted_probs.masked_fill(mask, 0.0)
    # probs = sorted_probs.scatter(-1, sorted_idx, sorted_probs)  # 散回原序
    # next_token = torch.multinomial(probs, num_samples=1)
    raise NotImplementedError

# 验证：p→0 时应趋近 argmax；分布尖锐时 nucleus 小

## 小结
- 截断条件易写反：要保留的是"累积**未**超过 p 的 + 第一个让累积超过 p 的"，等价于把 `cum - prob > p` 的置零。
- TopP 对长尾更鲁棒，是开放域生成（故事/对话）的默认选择；代码/事实类任务常用贪心或低温度。
- 工程上常 `temperature → topk → topp` 串联。

## ✅ 测试验证

In [ ]:
# 验证 Top-P (nucleus) 采样
import torch
import torch.nn.functional as F

logits = torch.randn(100)
P = 0.9

# Top-P: 按概率降序累加，直到累积概率 >= P，保留这些 token
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_indices = probs.sort(descending=True)
cumsum = sorted_probs.cumsum(dim=-1)

# 找到累积概率首次 >= P 的位置
cutoff = (cumsum <= P).sum().item() + 1  # +1 包含首次超过 P 的那个
nucleus = sorted_indices[:cutoff]

# 验证: 保留的 token 累积概率 >= P
nucleus_probs = probs[nucleus]
assert nucleus_probs.sum() >= P - 1e-6, f"nucleus sum {nucleus_probs.sum()} < P={P}"

# 验证: 去掉最小的那个就 < P
if cutoff > 1:
    nucleus_minus = nucleus[:-1]
    assert probs[nucleus_minus].sum() < P, "removing smallest should make sum < P"

print(f"✅ TopP 测试通过: nucleus 累积概率 {nucleus_probs.sum().item():.4f} >= P={P}")
